In [ ]:
import cv2
import os
import sys
import pandas as pd
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.impute import SimpleImputer
from sklearn.utils.class_weight import compute_class_weight
from sklearn.metrics import classification_report, mean_squared_error, mean_absolute_error
import tensorflow as tf
from tensorflow.keras import layers, models
import numpy as np
import matplotlib.pyplot as plt
def load_images(image_dir, data):
    labels = []
    images = []

    dirs = [image_dir + '/CS', image_dir + '/Healthy']
    for idx, image_dir in enumerate(dirs):
        for filename in sorted(os.listdir(image_dir)):
            if filename.endswith('.png'):
                # Decode filename to extract sequence number, gender, and age
                # print(filename)
                seq_number = int(filename[:4])  # First 4 digits: sequence number

                # Find the corresponding row in the dataset
                row = data[data['Number'] == seq_number]

                if row.empty:
                    print(f"No matching row found for {filename}")
                    continue

                # Drop the Disease Classification column to use all other columns as label
                label_row = row.drop(columns=['Disease classification: 1. Cervical spondylosis; 2. Healthy']).iloc[0]

                labels.append(label_row.values)

                img_path = os.path.join(image_dir, filename)
                image = cv2.imread(img_path)
                image = cv2.resize(image, (224, 224))
                images.append(image)

    return np.array(images)/255.0, np.array(labels)  # Normalize images

# File paths to the dataset and image directories
file_path = '/Users/srivatsavkannan/Datasets/C-Spine Xray/datasets.xlsx'
data = pd.read_excel(file_path, header=1).dropna()

train_image_dir = '/Users/srivatsavkannan/Datasets/FinalCervicalDataset/Train_Org_Aug'
val_image_dir = '/Users/srivatsavkannan/Datasets/FinalCervicalDataset/Val_Org_Aug'


# Load training and validation images and labels
X_train, y_train = load_images(train_image_dir, data)
print("Train Done..")

X_val, y_val = load_images(val_image_dir, data)


print(X_train.shape)
print(X_val.shape)
print(y_train.shape)
print(y_val.shape)

In [ ]:
checkpoint_dir = "checkpointsk/"
def build_yolo(input_shape, output_dim):
    inputs = layers.Input(shape=input_shape)

    # Feature extraction backbone (like YOLO's Darknet)
    x = layers.Conv2D(32, (3, 3), activation='relu', padding='same')(inputs)
    x = layers.MaxPooling2D((2, 2))(x)
    x = layers.Conv2D(64, (3, 3), activation='relu', padding='same')(x)
    x = layers.MaxPooling2D((2, 2))(x)
    x = layers.Conv2D(128, (3, 3), activation='relu', padding='same')(x)
    x = layers.MaxPooling2D((2, 2))(x)

    # Detection head for bounding box prediction
    bounding_box_head = layers.Flatten()(x)
    bounding_box_head = layers.Dense(128, activation='relu')(bounding_box_head)
    bounding_box_head = layers.Dense(output_dim, activation='linear', name='bounding_box')(bounding_box_head)

    model = models.Model(inputs=inputs, outputs=[bounding_box_head])
    return model


# Build the model
input_shape = (224, 224, 3)  # Image size with 3 channels (RGB)
output_dim = y_train.shape[1]  # Number of features to predict
model = build_yolo(input_shape, output_dim)

# Compile the model
model.compile(optimizer='adam',
              loss='mse',  # Mean Squared Error for regression
              metrics=['mae'])  # Mean Absolute Error

# Train the model
early_stop = tf.keras.callbacks.EarlyStopping(monitor='val_loss', patience=50, restore_best_weights=True)
checkpoint_callback = tf.keras.callbacks.ModelCheckpoint(
    filepath=os.path.join(checkpoint_dir, "model_epoch_{epoch:02d}.keras"),
    monitor='val_loss',  # Monitor validation accuracy
    save_best_only=True,  # Save only the model with the best val_accuracy
    save_freq='epoch',  # Save at the end of every epoch
    mode='min',  # Save model when val_loss improves
    verbose=1  # Print messages when saving
)
history = model.fit(X_train, y_train,
                    validation_data=(X_val, y_val),
                    epochs=50,
                    batch_size=32,
                    callbacks=[early_stop, checkpoint_callback],
                    verbose=1)

# Evaluate the model
train_loss, train_mae = model.evaluate(X_train, y_train, verbose=0)
test_loss, test_mae = model.evaluate(X_val, y_val, verbose=0)

print(f"Training Loss: {train_loss}, Training MAE: {train_mae}")
print(f"Testing Loss: {test_loss}, Testing MAE: {test_mae}")

# Predict on test data
predictions = model.predict(X_val)

# Print sample predictions and ground truth
num_samples = 5
indices = np.random.choice(len(X_val), num_samples, replace=False)
print(f"{'Sample':<8}{'Ground Truth':<30}{'Prediction':<30}")
print("=" * 70)
for i, idx in enumerate(indices):
    ground_truth = y_val[idx]
    predicted = predictions[idx]
    print(f"Sample {i + 1:<3} | GT: {ground_truth[:5]} | Pred: {predicted[:5]}")  # First 5 features

# Save the model
model.save("image_to_stats_model.keras")
print("Model saved as image_to_stats_model.keras")